In [2]:
import asyncio
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
import nest_asyncio
nest_asyncio.apply()  # For Jupyter/Colab

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import numpy as np
from dotenv import load_dotenv
import os
import pandas as pd

In [6]:

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

# Create multiple model instances for parallel calls
def create_gemini_model():
    return genai.GenerativeModel('gemini-2.5-flash')

In [17]:
def parse_scores(result_text):
    '''Parse multiple scores from judge response with improvement suggestions'''
    scores = {}
    try:
        lines = result_text.split('\n')
        for line in lines:
            line_lower = line.lower()
            
            if 'format accuracy:' in line_lower or 'format score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['format'] = float(score_str)
            elif 'reasoning quality:' in line_lower or 'reasoning score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['reasoning'] = float(score_str)
            elif 'answer quality:' in line_lower or 'quality score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                scores['answer_quality'] = float(score_str)
            elif 'answer accuracy:' in line_lower or 'accuracy score:' in line_lower:
                score_str = line.split(':')[1].strip().split('/')[0].strip()
                if score_str.upper() in ['N/A', 'N', 'NA', 'NONE']:
                    scores['answer_accuracy'] = None
                else:
                    scores['answer_accuracy'] = float(score_str)
        
        # Extract feedback
        feedback_start = result_text.find('Feedback:')
        reasoning_improvements_start = result_text.find('Reasoning Improvements:')
        answer_improvements_start = result_text.find('Answer Improvements:')
        
        if feedback_start != -1:
            # Extract feedback (ends at next section or end of text)
            end_pos = min([pos for pos in [reasoning_improvements_start, answer_improvements_start, len(result_text)] if pos > feedback_start])
            scores['feedback'] = result_text[feedback_start:end_pos].replace('Feedback:', '').strip()
        else:
            scores['feedback'] = result_text
        
        if reasoning_improvements_start != -1:
            end_pos = min([pos for pos in [answer_improvements_start, len(result_text)] if pos > reasoning_improvements_start])
            scores['reasoning_improvements'] = result_text[reasoning_improvements_start:end_pos].replace('Reasoning Improvements:', '').strip()
        else:
            scores['reasoning_improvements'] = 'N/A'
        
        if answer_improvements_start != -1:
            scores['answer_improvements'] = result_text[answer_improvements_start:].replace('Answer Improvements:', '').strip()
        else:
            scores['answer_improvements'] = 'N/A'
            
        return scores
    except Exception as e:
        print(f"Error parsing scores: {e}")
        print(f"Raw text: {result_text[:200]}")
        return None


def evaluate_with_ground_truth(input_text, model_response, ground_truth, domain, model):
    '''Evaluate response against ground truth with 4 separate scores + improvements'''
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with 4 separate scores.

GRADING SCALE (be strict and grounded):
0: Complete disaster - nonsensical, no coherent output
1-2: Very bad - severe issues, barely usable
3-5: Average quality - meets basic requirements but has notable issues
6-8: Decent but not great - good quality with minor issues
9: Great - excellent quality, minimal issues
10: Perfect - flawless execution

Question/Task: {input_text}

Ground Truth Answer: {ground_truth}

Model Response: {model_response}

Provide 4 separate scores using the scale above:

1. FORMAT ACCURACY (0-10):
   - 0: No tags, completely broken format
   - 1-2: Tags present but severely malformed (unclosed, wrong nesting)
   - 3-5: Tags present but has issues (extra tags, minor nesting problems, some parsing errors)
   - 6-8: Tags mostly correct with 1-2 minor issues (extra whitespace, slight formatting oddities)
   - 9: Nearly perfect format, one tiny issue
   - 10: Perfect - clean <reasoning>...</reasoning> and <answer>...</answer> tags, properly nested and closed

2. REASONING QUALITY (0-10):
   - 0: No reasoning or complete gibberish
   - 1-2: Incoherent reasoning, illogical steps
   - 3-5: Basic reasoning present but unclear, has logical gaps, or unnecessary verbosity
   - 6-8: Good logical flow with minor clarity issues or one missing step
   - 9: Clear, logical, well-structured with one tiny improvement possible
   - 10: Perfect step-by-step reasoning, crystal clear and complete

3. ANSWER QUALITY (0-10):
   - 0: No answer or completely unintelligible
   - 1-2: Answer present but poorly formatted or unclear
   - 3-5: Answer understandable but lacks clarity or completeness
   - 6-8: Well-formatted and clear answer with minor issues
   - 9: Excellent answer with one tiny improvement possible
   - 10: Perfect - clear, complete, well-formatted answer

4. ANSWER ACCURACY (0 or 1):
   - 0: Wrong answer (even if close)
   - 1: Correct answer (matches ground truth, equivalent formats accepted)

Then provide specific improvement suggestions:

REASONING IMPROVEMENTS (if reasoning quality < 9):
- List 2-3 specific, actionable bullet points
- Focus on: clarity, logical flow, missing steps, unnecessary verbosity
- If score is 9-10, write "None - reasoning is excellent"

ANSWER IMPROVEMENTS (if answer quality < 9 or accuracy = 0):
- List 2-3 specific, actionable bullet points
- Focus on: correctness, completeness, clarity, format
- If quality 9-10 and accuracy 1, write "None - answer is excellent"



Provide your evaluation in this exact format:
Format Accuracy: X/10
Reasoning Quality: X/10
Answer Quality: X/10
Answer Accuracy: 0/1
Feedback: [2-3 sentences explaining the scores, reference the grading scale]
Reasoning Improvements:
- [bullet point 1]
- [bullet point 2]
- [bullet point 3]
Answer Improvements:
- [bullet point 1]
- [bullet point 2]
- [bullet point 3]
'''
    
    try:
        response = model.generate_content(prompt)
        return parse_scores(response.text)
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None


def evaluate_creative(input_text, model_response, domain, model):
    '''Evaluate creative/generative responses with 3 scores + improvements'''
    
    domain_criteria = {
        'code': 'Code correctness, efficiency, handles edge cases, solves the problem',
        'creative_ideation': 'Creativity, relevance to prompt, feasibility, originality',
        'creative_writing': 'Writing quality, coherence, creativity, engagement',
        'summarization': 'Accuracy of main points, completeness, conciseness, clarity',
        'dialogue': 'Natural flow, context awareness, appropriate tone',
        'rewriting': 'Improvement over original, clarity, style',
        'conversation': 'Helpfulness, empathy, relevance, practical advice'
    }
    
    criteria = domain_criteria.get(domain, 'Quality, relevance, completeness, creativity')
    
    prompt = f'''
Evaluate this AI model's response for a {domain} task with 3 separate scores.

GRADING SCALE (be strict and grounded):
0: Complete disaster - nonsensical, no coherent output
1-2: Very bad - severe issues, barely usable
3-5: Average quality - meets basic requirements but has notable issues
6-8: Decent but not great - good quality with minor issues
9: Great - excellent quality, minimal issues
10: Perfect - flawless execution

Task/Prompt: {input_text}

Model Response: {model_response}

Provide 3 separate scores using the scale above:

1. FORMAT ACCURACY (0-10):
   - 0: No tags, completely broken format
   - 1-2: Tags present but severely malformed (unclosed, wrong nesting)
   - 3-5: Tags present but has issues (extra tags, minor nesting problems, some parsing errors)
   - 6-8: Tags mostly correct with 1-2 minor issues (extra whitespace, slight formatting oddities)
   - 9: Nearly perfect format, one tiny issue
   - 10: Perfect - clean <reasoning>...</reasoning> and <answer>...</answer> tags, properly nested and closed

2. REASONING QUALITY (0-10):
   - 0: No reasoning or complete gibberish
   - 1-2: Incoherent reasoning, fails to explain approach
   - 3-5: Basic reasoning present but lacks depth, clarity, or has gaps in logic
   - 6-8: Good explanation of thought process with minor clarity issues
   - 9: Excellent reasoning with one tiny improvement possible
   - 10: Perfect - insightful, clear, comprehensive reasoning

3. ANSWER QUALITY (0-10):
   For {domain}: {criteria}
   - 0: No answer or complete failure
   - 1-2: Severe quality issues, doesn't meet basic requirements
   - 3-5: Meets basic requirements but has notable issues in {criteria.lower()}
   - 6-8: Good quality with minor issues
   - 9: Excellent with one tiny improvement possible
   - 10: Perfect execution

Note: No accuracy score for {domain} as there's no single correct answer.

Then provide specific improvement suggestions:

REASONING IMPROVEMENTS (if reasoning quality < 9):
- List 2-3 specific, actionable bullet points
- Focus on: clarity, depth of thinking, approach quality, logical structure
- If score is 9-10, write "None - reasoning is excellent"

ANSWER IMPROVEMENTS (if answer quality < 9):
- List 2-3 specific, actionable bullet points
- Focus on: {criteria.lower()}
- If score is 9-10, write "None - answer is excellent"


Provide your evaluation in this exact format:
Format Accuracy: X/10
Reasoning Quality: X/10
Answer Quality: X/10
Answer Accuracy: N/A
Feedback: [2-3 sentences explaining the scores, reference the grading scale]
Reasoning Improvements:
- [bullet point 1]
- [bullet point 2]
- [bullet point 3]
Answer Improvements:
- [bullet point 1]
- [bullet point 2]
- [bullet point 3]
'''
    
    try:
        response = model.generate_content(prompt)
        scores = parse_scores(response.text)
        if scores:
            scores['answer_accuracy'] = None
        return scores
    except Exception as e:
        print(f"Error evaluating: {e}")
        return None


class AsyncGeminiEvaluator:
    def __init__(self, max_concurrent=50):
        """
        max_concurrent: Number of parallel API calls (Gemini allows high concurrency)
        """
        self.max_concurrent = max_concurrent
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.executor = ThreadPoolExecutor(max_workers=max_concurrent)
        
    async def evaluate_single_async(self, idx, row, model):
        """Async wrapper for single evaluation"""
        async with self.semaphore:
            loop = asyncio.get_event_loop()
            
            # Run blocking Gemini call in thread pool
            result = await loop.run_in_executor(
                self.executor,
                self._evaluate_blocking,
                idx, row, model
            )
            return result
    
    def _evaluate_blocking(self, idx, row, model):
        """Blocking evaluation function"""
        try:
            domain = row['domain']
            
            # Determine evaluation strategy
            if domain in GROUND_TRUTH_DOMAINS:
                scores = evaluate_with_ground_truth(
                    input_text=row['question'],
                    model_response=row['model_response'],
                    ground_truth=row['answer'],
                    domain=domain,
                    model=model
                )
            elif domain in CREATIVE_DOMAINS:
                scores = evaluate_creative(
                    input_text=row['question'],
                    model_response=row['model_response'],
                    domain=domain,
                    model=model
                )
            else:
                scores = evaluate_creative(
                    input_text=row['question'],
                    model_response=row['model_response'],
                    domain=domain,
                    model=model
                )
            
            return idx, scores
            
        except Exception as e:
            print(f"\nError evaluating row {idx}: {e}")
            return idx, None
    
    async def evaluate_all(self, eval_df):
        """Evaluate all rows concurrently"""
        # Create tasks for all rows
        tasks = []
        models = [create_gemini_model() for _ in range(self.max_concurrent)]
        
        for idx, row in eval_df.iterrows():
            model = models[idx % len(models)]
            task = self.evaluate_single_async(idx, row, model)
            tasks.append(task)
        
        # Run all tasks concurrently with progress
        results = []
        print(f"\nEvaluating {len(tasks)} responses with {self.max_concurrent} concurrent calls...")
        
        # Process in chunks to show progress
        chunk_size = 100
        for i in range(0, len(tasks), chunk_size):
            chunk_tasks = tasks[i:i+chunk_size]
            chunk_results = await asyncio.gather(*chunk_tasks)
            results.extend(chunk_results)
            print(f"  Progress: {len(results)}/{len(tasks)} ({len(results)/len(tasks)*100:.1f}%)")
        
        self.executor.shutdown()
        return results


# Define domain categories
GROUND_TRUTH_DOMAINS = {'math', 'science_reasoning', 'code', 'knowledge', 
                        'commonsense_reasoning', 'reading_comprehension'}
CREATIVE_DOMAINS = {'creative_writing', 'creative_ideation', 'summarization', 
                   'dialogue', 'rewriting', 'conversation'}


def evaluate_model(eval_df):
    """Main evaluation function with async processing"""
    import time
    start_time = time.time()
    
    # Create evaluator with high concurrency
    evaluator = AsyncGeminiEvaluator(max_concurrent=50)
    
    # Run async evaluation
    results = asyncio.run(evaluator.evaluate_all(eval_df))
    
    # Process results
    format_scores = [0] * len(eval_df)
    reasoning_scores = [0] * len(eval_df)
    answer_quality_scores = [0] * len(eval_df)
    answer_accuracy_scores = [None] * len(eval_df)
    feedbacks = [''] * len(eval_df)
    reasoning_improvements = [''] * len(eval_df)
    answer_improvements = [''] * len(eval_df)
    
    for idx, scores in results:
        if scores:
            format_scores[idx] = scores.get('format', 0)
            reasoning_scores[idx] = scores.get('reasoning', 0)
            answer_quality_scores[idx] = scores.get('answer_quality', 0)
            answer_accuracy_scores[idx] = scores.get('answer_accuracy')
            feedbacks[idx] = scores.get('feedback', 'Evaluation completed')
            reasoning_improvements[idx] = scores.get('reasoning_improvements', 'N/A')
            answer_improvements[idx] = scores.get('answer_improvements', 'N/A')
        else:
            feedbacks[idx] = "Evaluation failed"
            reasoning_improvements[idx] = "N/A"
            answer_improvements[idx] = "N/A"
    
    # Add to dataframe
    eval_df['format_accuracy'] = format_scores
    eval_df['reasoning_quality'] = reasoning_scores
    eval_df['answer_quality'] = answer_quality_scores
    eval_df['answer_accuracy'] = answer_accuracy_scores
    eval_df['judge_feedback'] = feedbacks
    eval_df['reasoning_improvements'] = reasoning_improvements
    eval_df['answer_improvements'] = answer_improvements
    
    # Calculate overall scores
    overall_scores = []
    for i in range(len(eval_df)):
        if format_scores[i] == 0:
            overall_scores.append(0)
        elif answer_accuracy_scores[i] is not None:
            score = (format_scores[i] * 0.1 + 
                    reasoning_scores[i] * 0.2 + 
                    answer_quality_scores[i] * 0.2 + 
                    answer_accuracy_scores[i]*10 * 0.5)
            overall_scores.append(score)
        else:
            score = (format_scores[i] * 0.15 + 
                    reasoning_scores[i] * 0.35 + 
                    answer_quality_scores[i] * 0.5)
            overall_scores.append(score)
    
    eval_df['overall_score'] = overall_scores
    
    elapsed = time.time() - start_time
    print(f"\n✓ Evaluation complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
    print(f"  Throughput: {len(eval_df)/elapsed:.1f} evaluations/second")
    
    # Print statistics
    print(f"\nOverall Statistics:")
    print(f"  Format Accuracy: {np.mean([s for s in format_scores if s > 0]):.2f}/10")
    print(f"  Reasoning Quality: {np.mean([s for s in reasoning_scores if s > 0]):.2f}/10")
    print(f"  Answer Quality: {np.mean([s for s in answer_quality_scores if s > 0]):.2f}/10")
    
    accuracy_valid = [s for s in answer_accuracy_scores if s is not None and s > 0]
    if accuracy_valid:
        print(f"  Answer Accuracy (verifiable domains): {np.mean(accuracy_valid):.2f}/10")
    
    print(f"  Overall Weighted Score: {np.mean([s for s in overall_scores if s > 0]):.2f}/10")
    
    # Analyze improvement patterns
    print("\n\nCommon Improvement Areas:")
    reasoning_needs_improvement = sum(1 for s in reasoning_scores if 0 < s < 9)
    answer_needs_improvement = sum(1 for s in answer_quality_scores if 0 < s < 9)
    print(f"  Reasoning needs improvement: {reasoning_needs_improvement}/{len(eval_df)} samples ({reasoning_needs_improvement/len(eval_df)*100:.1f}%)")
    print(f"  Answer needs improvement: {answer_needs_improvement}/{len(eval_df)} samples ({answer_needs_improvement/len(eval_df)*100:.1f}%)")
    
    return eval_df

In [ ]:
validation_sample = pd.read_csv("..//evaluation//eval_data//eval_results_benchmarkoos_v5.csv")
valid_eval = evaluate_model(eval_df=validation_sample)

# View improvement suggestions for low-scoring samples
low_reasoning = valid_eval[valid_eval['reasoning_quality'] < 3]
print("\nSamples with low reasoning scores:")
for idx, row in low_reasoning.head(5).iterrows():
    print(f"\nQuestion: {row['question'][:100]}...")
    print(f"Reasoning Score: {row['reasoning_quality']}/10")
    print(f"Improvements:\n{row['reasoning_improvements']}")


Evaluating 3260 responses with 50 concurrent calls...
Error evaluating: Invalid operation: The `response.parts` quick accessor requires a single candidate, but but `response.candidates` is empty.
This appears to be caused by a blocked prompt, see `response.prompt_feedback`: block_reason: PROHIBITED_CONTENT



Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000021C377C8640> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000021C377C8640> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000021C377C8640> is already entered
Exception in callback Task.__step(

  Progress: 100/3260 (3.1%)


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000021C377C8640> is already entered


Error evaluating: Invalid operation: The `response.parts` quick accessor requires a single candidate, but but `response.candidates` is empty.
This appears to be caused by a blocked prompt, see `response.prompt_feedback`: block_reason: PROHIBITED_CONTENT



Task was destroyed but it is pending!
task: <Task pending name='Task-819' coro=<_async_in_context.<locals>.run_in_context() done, defined at d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-820' coro=<Kernel.shell_main() running at d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\ipykernel\kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Python312\Lib\threading.py:293: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  self._waiters = _deque()
Task was destroyed but it is pending!
task: <Task pending name='Task-820' coro=<Kernel.shell_main() running at d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\ipykernel\kernelbase.py:590> cb=[Task.__wakeup()]>


  Progress: 200/3260 (6.1%)
  Progress: 300/3260 (9.2%)
  Progress: 400/3260 (12.3%)
  Progress: 500/3260 (15.3%)
  Progress: 600/3260 (18.4%)
  Progress: 700/3260 (21.5%)
  Progress: 800/3260 (24.5%)
  Progress: 900/3260 (27.6%)
  Progress: 1000/3260 (30.7%)
  Progress: 1100/3260 (33.7%)
  Progress: 1200/3260 (36.8%)
  Progress: 1300/3260 (39.9%)
  Progress: 1400/3260 (42.9%)
  Progress: 1500/3260 (46.0%)
  Progress: 1600/3260 (49.1%)
  Progress: 1700/3260 (52.1%)
  Progress: 1800/3260 (55.2%)
  Progress: 1900/3260 (58.3%)
  Progress: 2000/3260 (61.3%)
  Progress: 2100/3260 (64.4%)
  Progress: 2200/3260 (67.5%)
  Progress: 2300/3260 (70.6%)
  Progress: 2400/3260 (73.6%)
  Progress: 2500/3260 (76.7%)
  Progress: 2600/3260 (79.8%)
  Progress: 2700/3260 (82.8%)
  Progress: 2800/3260 (85.9%)
  Progress: 2900/3260 (89.0%)
  Progress: 3000/3260 (92.0%)
  Progress: 3100/3260 (95.1%)
  Progress: 3200/3260 (98.2%)
Error evaluating: Invalid operation: The `response.text` quick accessor requires

In [19]:
valid_eval.to_excel("..//evaluation//results//benchmark_eval_modelrunV5.xlsx")

In [20]:
valid_eval_sample=valid_eval[['domain','question','model_response','reasoning_quality','answer_quality','reasoning_improvements','answer_improvements']].sample(800)
valid_eval_sample.to_csv("..//evaluation//results//benchmark_eval_modelrunV5_sampleforai.csv",index=False)

Cleaner functions

In [ ]:
# import google.generativeai as genai
# from concurrent.futures import ThreadPoolExecutor
# import time
# from tqdm import tqdm

# # Safety settings to reduce blocking
# SAFETY_SETTINGS = [
#     {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
#     {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
#     {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
#     {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
# ]

# def create_gemini_model():
#     return genai.GenerativeModel(
#         'gemini-2.5-flash',
#         safety_settings=SAFETY_SETTINGS
#     )

# # YOUR EXACT ORIGINAL FUNCTION
# def parse_scores(result_text):
#     '''Parse multiple scores from judge response'''
#     scores = {}
#     try:
#         lines = result_text.split('\n')
#         for line in lines:
#             line_lower = line.lower()
            
#             if 'format accuracy:' in line_lower or 'format score:' in line_lower:
#                 score_str = line.split(':')[1].strip().split('/')[0].strip()
#                 scores['format'] = float(score_str)
#             elif 'reasoning quality:' in line_lower or 'reasoning score:' in line_lower:
#                 score_str = line.split(':')[1].strip().split('/')[0].strip()
#                 scores['reasoning'] = float(score_str)
#             elif 'answer quality:' in line_lower or 'quality score:' in line_lower:
#                 score_str = line.split(':')[1].strip().split('/')[0].strip()
#                 scores['answer_quality'] = float(score_str)
#             elif 'answer accuracy:' in line_lower or 'accuracy score:' in line_lower:
#                 score_str = line.split(':')[1].strip().split('/')[0].strip()
#                 if score_str.upper() in ['N/A', 'N', 'NA', 'NONE']:
#                     scores['answer_accuracy'] = None
#                 else:
#                     scores['answer_accuracy'] = float(score_str)
        
#         feedback_start = result_text.find('Feedback:')
#         if feedback_start != -1:
#             scores['feedback'] = result_text[feedback_start:].replace('Feedback:', '').strip()
#         else:
#             scores['feedback'] = result_text
            
#         return scores
#     except Exception as e:
#         print(f"Error parsing scores: {e}")
#         print(f"Raw text: {result_text[:200]}")
#         return None


# # YOUR EXACT ORIGINAL FUNCTION - only added blocked content check at the end
# def evaluate_with_ground_truth(input_text, model_response, ground_truth, domain):
#     '''Evaluate response against ground truth with 4 separate scores'''
    
#     prompt = f'''
# Evaluate this AI model's response for a {domain} task with 4 separate scores:

# Question/Task: {input_text}

# Ground Truth Answer: {ground_truth}

# Model Response: {model_response}

# Provide 4 separate scores (each 1-10):

# 1. FORMAT ACCURACY (1-10):
#    - Does response use <reasoning>...</reasoning> and <answer>...</answer> tags correctly?
#    - Are tags properly nested and closed?
#    - Is the structure clean and parseable?

# 2. REASONING QUALITY (1-10):
#    - Is the reasoning trace clear and logical?
#    - Does it show step-by-step thinking?
#    - Are intermediate steps correct?
#    - Is the reasoning helpful for understanding the solution?

# 3. ANSWER QUALITY (1-10):
#    - Is the answer well-formatted and complete?
#    - Is it clear and understandable?
#    - Does it directly address the question?

# 4. ANSWER ACCURACY (0 = Wrong/1 =Correct):
#    - Does the final answer match the ground truth?
#    - Is it mathematically/factually correct?
#    - If the answer is correct but is represented in a different format it's fine

# Provide your evaluation in this exact format:
# Format Accuracy: X/10
# Reasoning Quality: X/10
# Answer Quality: X/10
# Answer Accuracy: 0/1
# Feedback: [2-3 sentences explaining the scores]
# '''
    
#     try:
#         response = gemini_model.generate_content(prompt)
        
#         # Handle blocked content
#         if not response.candidates:
#             return {
#                 'format': 5, 'reasoning': 5, 'answer_quality': 5,
#                 'answer_accuracy': 0, 'feedback': 'Content blocked'
#             }
        
#         return parse_scores(response.text)
#     except Exception as e:
#         print(f"Error evaluating: {e}")
#         return None


# # YOUR EXACT ORIGINAL FUNCTION - only added blocked content check at the end
# def evaluate_creative(input_text, model_response, domain):
#     '''Evaluate creative/generative responses with 3 scores (no accuracy)'''
    
#     domain_criteria = {
#         'code': 'Code correctness, efficiency, handles edge cases, solves the problem',
#         'creative_ideation': 'Creativity, relevance to prompt, feasibility, originality',
#         'creative_writing': 'Writing quality, coherence, creativity, engagement',
#         'summarization': 'Accuracy of main points, completeness, conciseness, clarity'
#     }
    
#     criteria = domain_criteria.get(domain, 'Quality, relevance, completeness, creativity')
    
#     prompt = f'''
# Evaluate this AI model's response for a {domain} task with 3 separate scores:

# Task/Prompt: {input_text}

# Model Response: {model_response}

# Provide 3 separate scores (each 1-10):

# 1. FORMAT ACCURACY (1-10):
#    - Does response use <reasoning>...</reasoning> and <answer>...</answer> tags correctly?
#    - Are tags properly nested and closed?
#    - Is the structure clean and parseable?

# 2. REASONING QUALITY (1-10):
#    - Is the reasoning trace clear and logical?
#    - Does it explain the thought process well?
#    - Is the approach sound?
#    - Does reasoning add value to understanding the solution?

# 3. ANSWER QUALITY (1-10):
#    - {criteria}
#    - Is the answer complete and well-executed?
#    - Does it fulfill the prompt requirements?

# Note: No accuracy score for {domain} as there's no single correct answer.

# Provide your evaluation in this exact format:
# Format Accuracy: X/10
# Reasoning Quality: X/10
# Answer Quality: X/10
# Answer Accuracy: N/A
# Feedback: [2-3 sentences explaining the scores]
# '''
    
#     try:
#         response = gemini_model.generate_content(prompt)
        
#         # Handle blocked content
#         if not response.candidates:
#             return {
#                 'format': 5, 'reasoning': 5, 'answer_quality': 5,
#                 'answer_accuracy': None, 'feedback': 'Content blocked'
#             }
        
#         scores = parse_scores(response.text)
#         if scores:
#             scores['answer_accuracy'] = None
#         return scores
#     except Exception as e:
#         print(f"Error evaluating: {e}")
#         return None


# # Threading-based evaluator (replaces asyncio)
# class ThreadedGeminiEvaluator:
#     def __init__(self, max_workers=30):
#         self.max_workers = max_workers
#         self.executor = ThreadPoolExecutor(max_workers=max_workers)
        
#     def evaluate_single(self, idx, row):
#         try:
#             domain = row['domain']
            
#             if domain in GROUND_TRUTH_DOMAINS:
#                 scores = evaluate_with_ground_truth(
#                     input_text=row['input'],
#                     model_response=row['model_response'],
#                     ground_truth=row['ground_truth'],
#                     domain=domain
#                 )
#             elif domain in CREATIVE_DOMAINS:
#                 scores = evaluate_creative(
#                     input_text=row['input'],
#                     model_response=row['model_response'],
#                     domain=domain
#                 )
#             else:
#                 scores = evaluate_creative(
#                     input_text=row['input'],
#                     model_response=row['model_response'],
#                     domain=domain
#                 )
            
#             return idx, scores
#         except Exception as e:
#             print(f"\nError evaluating row {idx}: {e}")
#             return idx, None
    
#     def evaluate_all(self, eval_df):
#         futures = []
#         for idx, row in eval_df.iterrows():
#             future = self.executor.submit(self.evaluate_single, idx, row)
#             futures.append(future)
        
#         results = []
#         print(f"\nEvaluating {len(futures)} responses with {self.max_workers} concurrent threads...")
        
#         for future in tqdm(futures, desc="Evaluating"):
#             try:
#                 result = future.result(timeout=60)
#                 results.append(result)
#             except:
#                 results.append((len(results), None))
        
#         self.executor.shutdown(wait=True)
#         return results


# GROUND_TRUTH_DOMAINS = {'math', 'science_reasoning', 'code', 'knowledge', 
#                         'commonsense_reasoning', 'reading_comprehension'}
# CREATIVE_DOMAINS = {'creative_writing', 'creative_ideation', 'summarization', 
#                    'dialogue', 'rewriting', 'conversation'}


# def evaluate_model(eval_df):
#     start_time = time.time()
    
#     evaluator = ThreadedGeminiEvaluator(max_workers=30)
#     results = evaluator.evaluate_all(eval_df)
    
#     format_scores = []
#     reasoning_scores = []
#     answer_quality_scores = []
#     answer_accuracy_scores = []
#     feedbacks = []

#     for idx, scores in results:
#         if scores:
#             format_scores.append(scores.get('format', 0))
#             reasoning_scores.append(scores.get('reasoning', 0))
#             answer_quality_scores.append(scores.get('answer_quality', 0))
#             answer_accuracy_scores.append(scores.get('answer_accuracy', 0) if scores.get('answer_accuracy') is not None else None)
#             feedbacks.append(scores.get('feedback', 'Evaluation completed'))
#         else:
#             format_scores.append(0)
#             reasoning_scores.append(0)
#             answer_quality_scores.append(0)
#             answer_accuracy_scores.append(None)
#             feedbacks.append("Evaluation failed")

#     eval_df['format_accuracy'] = format_scores
#     eval_df['reasoning_quality'] = reasoning_scores
#     eval_df['answer_quality'] = answer_quality_scores
#     eval_df['answer_accuracy'] = answer_accuracy_scores
#     eval_df['judge_feedback'] = feedbacks

#     overall_scores = []
#     for i in range(len(eval_df)):
#         if format_scores[i] == 0:
#             overall_scores.append(0)
#         elif answer_accuracy_scores[i] is not None:
#             score = (format_scores[i] * 0.1 + 
#                     reasoning_scores[i] * 0.2 + 
#                     answer_quality_scores[i] * 0.2 + 
#                     answer_accuracy_scores[i]*10 * 0.5)
#             overall_scores.append(score)
#         else:
#             score = (format_scores[i] * 0.15 + 
#                     reasoning_scores[i] * 0.35 + 
#                     answer_quality_scores[i] * 0.5)
#             overall_scores.append(score)

#     eval_df['overall_score'] = overall_scores

#     elapsed = time.time() - start_time
#     print(f"\n✓ Evaluation complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    
#     print(f"\nOverall Statistics:")
#     print(f"  Format Accuracy: {np.mean([s for s in format_scores if s > 0]):.2f}/10")
#     print(f"  Reasoning Quality: {np.mean([s for s in reasoning_scores if s > 0]):.2f}/10")
#     print(f"  Answer Quality: {np.mean([s for s in answer_quality_scores if s > 0]):.2f}/10")

#     accuracy_valid = [s for s in answer_accuracy_scores if s is not None and s > 0]
#     if accuracy_valid:
#         print(f"  Answer Accuracy (verifiable domains): {np.mean(accuracy_valid):.2f}/10")

#     print(f"  Overall Weighted Score: {np.mean([s for s in overall_scores if s > 0]):.2f}/10")

#     return eval_df